In [7]:
from huggingface_hub import snapshot_download
import pandas as pd
import glob
import polars as pl
import os
from pathlib import Path
import zipfile
import shutil

CWD = Path(os.getcwd()).parent

https://huggingface.co/datasets/Zihan1004/FNSPID

In [ ]:
# 1. Download into ../data/FNSPID
snapshot_download(
    repo_id="Zihan1004/FNSPID",
    repo_type="dataset",
    local_dir=os.path.join(CWD, "data", "FNSPID")
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Stock_news/All_external.csv:   0%|          | 0.00/5.73G [00:00<?, ?B/s]

Stock_price/full_history.zip:   0%|          | 0.00/590M [00:00<?, ?B/s]

Stock_news/nasdaq_exteral_data.csv:   0%|          | 0.00/23.2G [00:00<?, ?B/s]

'/Users/armandhubler/Documents/coding_project/nlp_project/notebook/FNSPID'

In [ ]:
# Read csv file
df_nas = pl.scan_csv(
    os.path.join(CWD, "data", "FNSPID", "Stock_news", "nasdaq_exteral_data.csv")
)

# Sink file to parquet format for faster loading in the future
df_nas.sink_parquet(
    os.path.join(CWD, "data", "FNSPID", "Stock_news", "nasdaq_external_data.parquet")
)

# Delete the original csv file to save space
os.remove(os.path.join(CWD, "data", "FNSPID", "Stock_news", "nasdaq_exteral_data.csv"))

In [ ]:
# Read csv file
df_ext = pl.scan_csv(
    os.path.join(CWD, "data", "FNSPID", "Stock_news", "All_external.csv")
)

# Sink file to parquet format for faster loading in the future
df_ext.sink_parquet(
    os.path.join(CWD, "data", "FNSPID", "Stock_news", "All_external.parquet")
)

# Delete the original csv file to save space
os.remove(os.path.join(CWD, "data", "FNSPID", "Stock_news", "All_external.csv"))

In [ ]:
os.makedirs(os.path.join(CWD, "data", "Stock_news"), exist_ok=True)

# Move nasdaq_exteral_data.parquet to Stock_news folder
os.rename(
    os.path.join(CWD, "data", "FNSPID", "Stock_news", "nasdaq_external_data.parquet"),
    os.path.join(CWD, "data", "Stock_news", "nasdaq_external_data.parquet")
)

# Move All_external.parquet to Stock_news folder
os.rename(
    os.path.join(CWD, "data", "FNSPID", "Stock_news", "All_external.parquet"),
    os.path.join(CWD, "data", "Stock_news", "All_external.parquet")
)

In [ ]:
os.makedirs(os.path.join(CWD, "data", "Stock_price"), exist_ok=True)

os.rename(
    os.path.join(CWD, "data", "FNSPID", "Stock_price", "full_history.zip"),
    os.path.join(CWD, "data", "Stock_price", "full_history.zip")
)

In [ ]:
# Unzip the full_history.zip file
with zipfile.ZipFile(os.path.join(CWD, "data", "Stock_price", "full_history.zip"), 'r') as zip_ref:
    zip_ref.extractall(os.path.join(CWD, "data", "Stock_price"))

os.rename(
    os.path.join(CWD, "data", "Stock_price", "full_history"),
    os.path.join(CWD, "data", "full_history")
)

shutil.rmtree(os.path.join(CWD, "data", "Stock_price"))

os.rename(
    os.path.join(CWD, "data", "full_history"),
    os.path.join(CWD, "data", "Stock_price")
)

In [16]:
# Read all .csv files in Stock_price folder and concatenate them into a single parquet file
csv_files = glob.glob(os.path.join(CWD, "data", "Stock_price", "*.csv"))
df_list = []
for file in csv_files:
    df = pl.read_csv(file)
    expected_schema = {
        "date": pl.String,
        "volume": pl.Int64,
        "open": pl.Float64,
        "high": pl.Float64,
        "low": pl.Float64,
        "close": pl.Float64,
        "adj close": pl.Float64,
    }
    if df.schema != expected_schema:
        continue
    df = df.with_columns(pl.lit(file.split("/")[-1].split(".")[0]).alias("ticker"))
    df_list.append(df)
df_stock_price = pl.concat(df_list)
df_stock_price.write_parquet(os.path.join(CWD, "data", "Stock_price", "stock_price.parquet"))

In [17]:
# Delete all .csv files in Stock_price folder
for file in csv_files:
    os.remove(file)

In [ ]:
shutil.rmtree(os.path.join(CWD, "notebook", "FNSPID"))